In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
import ast


In [2]:
ads_base = pd.read_csv('../data/ads.csv')
users_base = pd.read_csv('../data/users.csv')

users= users_base.copy()
ads = ads_base.copy()

In [3]:
users.head(1)

,user_id,age,gender,state,city,preferences
0,0,20,F,MG,Contagem,"[('Game_Streaming', 0.41710305106179), ('Mobil..."


In [4]:
ads.head(1)

,ad_id,state_target,city_target,age_minimum_target,age_maximum_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
0,0,Any,Any,0,18,F,"{'Counter_Strike': 0.2199486778790079, 'League...",21.519743,10759.871442,0.850033,0.013965


In [5]:
type(ads.categories)

pandas.core.series.Series

In [6]:
type(users.preferences)

pandas.core.series.Series

In [7]:
# CSV perde a informação de tipo, então precisamos converter as strings de volta para listas e para dicionários. Para isso, podemos usar a função ast.literal_eval, que avalia uma string como uma expressão Python e retorna o objeto correspondente.

users["preferences"] = users["preferences"].apply(ast.literal_eval)
ads['categories'] = ads['categories'].apply(ast.literal_eval)

# Features Categorias

Gerar as categorias com seus pesos

OBS: isso foi feito porque para trabalhar com a abordagem de vetores, ADS e Users devem possuir mesma dimensões

Depois desenvolver uma forma mais inteligente para não fazer à mão

In [8]:
all_categories  = [
"FPS",
"Battle_Royale",
"MOBA",
"RPG",
"MMORPG",
"Strategy",
"Simulation",
"Sports_Games",
"Racing",
"Indie_Games",
"Retro_Gaming",
"Mobile_Gaming",
"PC_Gaming",
"Console_Gaming",
"Game_Development",
"Game_Streaming",
"Esports",
"Game_Hardware",
"VR_Gaming",
"Open_World",
"League_of_Legends",
"Fortnite",
"Minecraft",
"Counter_Strike",
"World_of_Warcraft"
]

In [9]:
# enumerar as categorias para criar um índice de categoria, que será usado para construir os vetores de características dos usuários e dos anúncios.
category_index = {cat: i for i, cat in enumerate(all_categories)}


In [10]:
category_index

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [11]:
# função para construir o vetor de características do usuário a partir de suas preferências, usando o índice de categoria para mapear as categorias para posições no vetor.

def build_user_vector(preferences, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in preferences:
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [12]:
# fazendo um teste para ver se a função build_user_vector está funcionando corretamente, construindo o vetor de características do primeiro usuário do dataset.
teste = build_user_vector(users.iloc[0]['preferences'], category_index)
display(teste)
display(users.iloc[0]['preferences'])
display(category_index)

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.58289695, 0.        , 0.        , 0.        ,
       0.41710305, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

[('Game_Streaming', 0.41710305106179), ('Mobile_Gaming', 0.58289694893821)]

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [13]:
len(teste)

25

In [14]:
# função análoga para o ads
# feito isso pq o ads foi salvo como dict e user como list

def build_ad_vector(categories, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in categories.items():
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [15]:
# teste para ads
teste_ad = build_ad_vector(ads.iloc[0]['categories'], category_index)
display(teste_ad)
display(ads.iloc[0]['categories'])


array([0.        , 0.04975271, 0.05743314, 0.        , 0.0592876 ,
       0.2054784 , 0.        , 0.1400657 , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.14804513, 0.07857752, 0.        , 0.        , 0.        ,
       0.04141114, 0.        , 0.        , 0.21994868, 0.        ])

{'Counter_Strike': 0.2199486778790079,
 'League_of_Legends': 0.04141113658357547,
 'Strategy': 0.20547839516888508,
 'MOBA': 0.0574331396610691,
 'Game_Streaming': 0.14804512754084037,
 'Esports': 0.07857752038331933,
 'Sports_Games': 0.14006569638784216,
 'Battle_Royale': 0.049752709933774945,
 'MMORPG': 0.05928759646168552}

In [16]:
# aplcar as funções para construir os vetores de características para todos os usuários e anúncios, e armazenar esses vetores em novas colunas nos dataframes.

users['feature_vector'] = users['preferences'].apply(lambda prefs: build_user_vector(prefs, category_index))
ads['feature_vector'] = ads['categories'].apply(lambda cats: build_ad_vector(cats, category_index))

In [17]:
users.head(1)

,user_id,age,gender,state,city,preferences,feature_vector
0,0,20,F,MG,Contagem,"[(Game_Streaming, 0.41710305106179), (Mobile_G...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Features demográficas

Uma estratégia é adicionar essas features ao vetor, só que aumenta bastante a dimensionalidade.

Inicialmente, vamos fazer one-hot encoding, mas é melhor pensar em outras estratégias
-   nessas situações, há 25 cidades distintas

In [18]:
ALL_CITIES = sorted(
    set(users["city"].unique()) |
    set(ads["city_target"].unique())
)

In [19]:
ALL_STATES = sorted(
    set(users["state"].unique()) | 
    set(ads["state_target"].unique())
)

In [20]:
def one_hot_city(city, all_cities):

    vector = {f"city_{c}": 0 for c in all_cities}

    if city in all_cities:
        vector[f"city_{city}"] = 1

    return vector

In [21]:
def one_hot_state(state, all_states):

    vector = {f"state_{s}": 0 for s in all_states}

    if state in all_states:
        vector[f"state_{state}"] = 1

    return vector

In [22]:
city_features = users["city"].apply(
    lambda x: pd.Series(one_hot_city(x, ALL_CITIES))
)

city_features_ads = ads["city_target"].apply(
    lambda x: pd.Series(one_hot_city(x, ALL_CITIES))
)

In [23]:
state_features = users["state"].apply(
    lambda x: pd.Series(one_hot_state(x, ALL_STATES))
)
state_features_ads = ads["state_target"].apply(
    lambda x: pd.Series(one_hot_state(x, ALL_STATES))
)

In [24]:
users = pd.concat([users, city_features], axis=1)
ads = pd.concat([ads, city_features_ads], axis=1)

In [25]:
users = pd.concat([users, state_features], axis=1)
ads = pd.concat([ads, state_features_ads], axis=1)

### Feature genero

In [26]:
# One-hot encoding para Genero
gender_dummies = pd.get_dummies(users["gender"], prefix="gender")
users = pd.concat([users, gender_dummies], axis=1)

gender_dummies_ads = pd.get_dummies(ads["gender_target"], prefix="gender")
ads = pd.concat([ads, gender_dummies_ads], axis=1)

### Feature Idade

In [27]:
# a idade targe só tem uma idade fixa, então faz sentido normalizar.
# isso vai acontecer ou o anúncio vai ter uma faixa de valores??

In [28]:
age_min = users['age'].min()
age_max = users['age'].max()

users['age_norm'] = (users['age'] - age_min) / (age_max - age_min)

ads['age_min_norm'] = (ads['age_minimum_target'] - age_min) / (age_max - age_min)
ads['age_max_norm'] = (ads['age_maximum_target'] - age_min) / (age_max - age_min)

# Gerar os vetores finais com cidade/estado

- vai fazer a similaridade tudo junto ou vai separar??

In [29]:
# definir ordem fixa das features

CATEGORY_COLUMNS = all_categories

CITY_COLUMNS = [f"city_{c}" for c in ALL_CITIES]

STATE_COLUMNS = [f"state_{s}" for s in ALL_STATES]

GENDER_COLUMNS = ["gender_F", "gender_M"]

In [30]:
# transformar features_vector em colunas separadas
category_features_users = pd.DataFrame(
    users["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

category_features_ads = pd.DataFrame(
    ads["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

users = pd.concat([users, category_features_users], axis=1)
ads = pd.concat([ads, category_features_ads], axis=1)

In [31]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.417103,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.089291,0.000000,0.910709,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.595983,0.159594,0.000000,0.000000,0.000000,0.244423,0.000000,0.0
3,0.0,0.000000,0.229662,0.0,0.0,0.0000,0.000000,0.269283,0.108461,0.000000,...,0.045426,0.0,0.316789,0.000000,0.000000,0.000000,0.000000,0.000000,0.030378,0.0
4,0.0,0.037115,0.241853,0.0,0.0,0.1263,0.000000,0.000000,0.013749,0.000000,...,0.000000,0.0,0.000000,0.127018,0.012052,0.026839,0.000000,0.000000,0.121998,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.204996,0.000000,0.000000,...,0.000000,0.0,0.000000,0.795004,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9996,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.650203,0.000000,0.000000,0.000000,...,0.164653,0.0,0.000000,0.000000,0.000000,0.000000,0.185145,0.000000,0.000000,0.0
9997,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.191319,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9998,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.202205,...,0.000000,0.0,0.000000,0.167820,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [32]:
users.iloc[0].preferences

[('Game_Streaming', 0.41710305106179), ('Mobile_Gaming', 0.58289694893821)]

In [33]:
users[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.000000
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.000000
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.582897
PC_Gaming            0.000000
Console_Gaming       0.000000
Game_Development     0.000000
Game_Streaming       0.417103
Esports              0.000000
Game_Hardware        0.000000
VR_Gaming            0.000000
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.000000
Minecraft            0.000000
Counter_Strike       0.000000
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [34]:
# garantir que todas as colunas de cidade, estado e gênero estejam presentes em ambos os dataframes, preenchendo com zeros onde necessário
for col in GENDER_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

In [35]:
for col in CITY_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

for col in STATE_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

In [36]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS +
    CITY_COLUMNS +
    STATE_COLUMNS +
    GENDER_COLUMNS
)

In [37]:
user_matrix = users[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [38]:
user_matrix

array([[0.0, 0.0, 0.0, ..., 0, True, False],
       [0.0, 0.0, 0.0, ..., 0, True, False],
       [0.0, 0.0, 0.0, ..., 0, False, True],
       ...,
       [0.0, 0.0, 0.0, ..., 1, True, False],
       [0.0, 0.0, 0.0, ..., 0, True, False],
       [0.0, 0.9430494520493434, 0.0, ..., 1, True, False]], dtype=object)

In [39]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [40]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.31917536, 0.        , 0.009381  , ..., 0.        , 0.        ,
        0.53347608],
       [0.28765744, 0.        , 0.28115972, ..., 0.00662745, 0.69123546,
        0.2552428 ],
       [0.        , 0.26956591, 0.29693717, ..., 0.40674805, 0.        ,
        0.26956591],
       ...,
       [0.30687257, 0.        , 0.        , ..., 0.        , 0.02019713,
        0.27229269],
       [0.30267943, 0.        , 0.29584238, ..., 0.        , 0.72733302,
        0.26857205],
       [0.29901034, 0.        , 0.        , ..., 0.        , 0.02581253,
        0.25342585]])

## Validar os resultados

In [43]:
user_index = 0

user_age = users.iloc[user_index]["age"]

scores = similarity_matrix[user_index]

ads_filtered = ads[
    (
        (ads["age_minimum_target"] == 0) |
        (user_age >= ads["age_minimum_target"])
    ) &
    (
        (ads["age_maximum_target"] == 100) |
        (user_age <= ads["age_maximum_target"])
    )
]

filtered_scores = scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:10]]

In [44]:
top_ads

Index([12, 7, 186, 617, 72, 566, 184, 166, 891, 495], dtype='int64')

In [45]:
scores[top_ads]

array([0.93345881, 0.92355319, 0.84940375, 0.82165529, 0.80021412,
       0.80021412, 0.80021412, 0.80021412, 0.80021412, 0.80021412])

In [46]:
ads_base.iloc[top_ads]

,ad_id,state_target,city_target,age_minimum_target,age_maximum_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
12,12,MG,Contagem,0,100,F,"{'World_of_Warcraft': 0.1600209062833675, 'Rac...",15.037382,7518.691233,0.953438,0.037696
7,7,MG,Contagem,0,22,F,"{'MMORPG': 0.0007625867777573193, 'VR_Gaming':...",42.499571,42499.571083,0.611246,0.007780
186,186,MG,Contagem,0,32,F,"{'Counter_Strike': 0.17025094313462488, 'Open_...",34.477390,3447.738961,0.997289,0.030536
617,617,MG,Contagem,0,100,F,"{'World_of_Warcraft': 0.11661827000557698, 'MM...",29.006345,29006.345333,0.799902,0.013228
72,72,MG,Contagem,0,53,F,{'Racing': 1.0},15.000000,15000.000000,0.729680,0.036152
566,566,MG,Contagem,19,61,F,{'Esports': 0.9999999999999999},20.580347,2058.034670,0.689096,0.036645
184,184,MG,Contagem,0,50,F,{'RPG': 1.0},28.335946,28335.946280,0.158136,0.002321
166,166,MG,Contagem,0,100,F,{'Game_Development': 1.0},38.402163,38402.162879,0.983324,0.047306
891,891,MG,Contagem,0,100,F,{'Racing': 1.0},18.111040,1811.103992,0.881144,0.048888
495,495,MG,Contagem,0,100,F,{'PC_Gaming': 1.0},54.452716,54452.716368,0.546417,0.008067


In [47]:
users.loc[0]

user_id                                                                      0
age                                                                         20
gender                                                                       F
state                                                                       MG
city                                                                  Contagem
preferences                  [(Game_Streaming, 0.41710305106179), (Mobile_G...
feature_vector               [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
city_Any                                                                     0
city_Belo Horizonte                                                          0
city_Campinas                                                                0
city_Contagem                                                                1
city_Curitiba                                                                0
city_Duque de Caxias                                

# Gerar os vetores finais sem cidade/estado

In [48]:
users_no_location = users
ads_no_location = ads

In [49]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.417103,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.089291,0.000000,0.910709,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.595983,0.159594,0.000000,0.000000,0.000000,0.244423,0.000000,0.0
3,0.0,0.000000,0.229662,0.0,0.0,0.0000,0.000000,0.269283,0.108461,0.000000,...,0.045426,0.0,0.316789,0.000000,0.000000,0.000000,0.000000,0.000000,0.030378,0.0
4,0.0,0.037115,0.241853,0.0,0.0,0.1263,0.000000,0.000000,0.013749,0.000000,...,0.000000,0.0,0.000000,0.127018,0.012052,0.026839,0.000000,0.000000,0.121998,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.204996,0.000000,0.000000,...,0.000000,0.0,0.000000,0.795004,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9996,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.650203,0.000000,0.000000,0.000000,...,0.164653,0.0,0.000000,0.000000,0.000000,0.000000,0.185145,0.000000,0.000000,0.0
9997,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.191319,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9998,0.0,0.000000,0.000000,0.0,0.0,0.0000,0.000000,0.000000,0.000000,0.202205,...,0.000000,0.0,0.000000,0.167820,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [50]:
users_no_location.iloc[0].preferences

[('Game_Streaming', 0.41710305106179), ('Mobile_Gaming', 0.58289694893821)]

In [51]:
users_no_location[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.000000
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.000000
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.582897
PC_Gaming            0.000000
Console_Gaming       0.000000
Game_Development     0.000000
Game_Streaming       0.417103
Esports              0.000000
Game_Hardware        0.000000
VR_Gaming            0.000000
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.000000
Minecraft            0.000000
Counter_Strike       0.000000
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [52]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS
)

In [53]:
user_matrix = users_no_location[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads_no_location[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [54]:
user_matrix

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.24442316, 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.94304945, 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [55]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [56]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.22295422, 0.        , 0.0817967 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.03474122, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.83921318, 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.21080019,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.12852209, 0.        , 0.        , ..., 0.        , 0.1868337 ,
        0.        ]])

## Validar os resultados

In [57]:
user_index = 0

user_city = users_no_location.iloc[user_index]["city"]
user_gender = users_no_location.iloc[user_index]["gender"]
user_age = users_no_location.iloc[user_index]["age"]
user_state = users_no_location.iloc[user_index]["state"]

scores = similarity_matrix[user_index]

ads_filtered = ads_no_location[
    (ads_no_location["gender_target"].isin([user_gender, "Any"])) &
    (
        (ads_no_location["age_minimum_target"] == 0) |
        (user_age >= ads_no_location["age_minimum_target"])
    ) &
    (
        (ads_no_location["age_maximum_target"] == 100) |
        (user_age <= ads_no_location["age_maximum_target"])
    )
]

ads_filtered = ads_filtered[
    (ads_filtered["state_target"].isin([user_state, "Any"])) &
    (ads_filtered["city_target"].isin([user_city, "Any"]))
]

filtered_scores = scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:10]]

In [58]:
top_ads

Index([503, 765, 436, 190, 382, 180, 479, 778, 123, 12], dtype='int64')

In [59]:
scores[top_ads]

array([0.81323953, 0.81323953, 0.67331995, 0.65802037, 0.59324133,
       0.58046997, 0.57637641, 0.52449233, 0.50233628, 0.49543137])

In [60]:
ads_base.iloc[top_ads]

,ad_id,state_target,city_target,age_minimum_target,age_maximum_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
503,503,Any,Any,0,100,Any,{'Mobile_Gaming': 1.0},17.023388,1702.338751,0.766228,0.032687
765,765,Any,Any,0,100,Any,{'Mobile_Gaming': 1.0},31.389181,15694.590529,0.704523,0.016575
436,436,Any,Any,0,45,F,"{'Game_Hardware': 0.12236615254750695, 'Mobile...",15.244514,1524.451436,0.997689,0.009111
190,190,MG,Any,0,100,F,"{'League_of_Legends': 0.14763777061208175, 'MO...",15.000000,15000.000000,0.767060,0.026613
382,382,Any,Any,0,31,Any,"{'Mobile_Gaming': 0.3622767333403724, 'Console...",23.568283,11784.141572,0.815579,0.031870
180,180,Any,Any,0,22,Any,"{'Open_World': 0.0662483705736774, 'Game_Strea...",15.000000,1500.000000,0.828258,0.016704
479,479,Any,Any,0,44,F,"{'Mobile_Gaming': 0.5011570921049712, 'Simulat...",15.000000,1500.000000,0.854398,0.031436
778,778,Any,Any,0,100,F,"{'Game_Streaming': 0.6037094122313995, 'Consol...",15.483057,7741.528483,0.922708,0.025565
123,123,Any,Any,18,100,Any,"{'Game_Streaming': 0.5769493583569131, 'Minecr...",16.618966,1661.896625,0.730486,0.023203
12,12,MG,Contagem,0,100,F,"{'World_of_Warcraft': 0.1600209062833675, 'Rac...",15.037382,7518.691233,0.953438,0.037696


In [61]:
users.loc[0]

user_id                                                                      0
age                                                                         20
gender                                                                       F
state                                                                       MG
city                                                                  Contagem
preferences                  [(Game_Streaming, 0.41710305106179), (Mobile_G...
feature_vector               [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
city_Any                                                                     0
city_Belo Horizonte                                                          0
city_Campinas                                                                0
city_Contagem                                                                1
city_Curitiba                                                                0
city_Duque de Caxias                                

# **OBJETIVO : score final**

score =
interest_similarity
× demographic_score
× bid
× trust_score
× estimated_conversion_rate

NÃO ESQUECER: incluir as